# LinkedIn Job Scraper Documentation

## Project Overview
This notebook implements a comprehensive web scraping solution to extract job listings from LinkedIn. The primary objective is to collect at least 20 jobs from multiple specific domains and search criteria to build a diverse job database that can be matched against CVs/resumes for job recommendation or candidate matching purposes.

### Key Objectives:
1. **Diverse Job Collection**: Scrape jobs from 30+ different search URLs covering various domains including:
   - Software Development (C++, Full Stack, React, DevOps)
   - Data Science & AI (NLP, Data Analyst, AI/ML Engineer)
   - Engineering (Mechatronics, Robotics, Biomechanical)
   - Business & Finance (Investment Banking, Finance, Business Development)
   - Marketing & Sales (Digital Marketing, Marketing Executive, Sales)
   - HR & Operations (HR Coordinator, Recruitment Specialist)
   - Internships across multiple fields

2. **Geographic Focus**: All searches are filtered for Egypt (geoId=106155005) to create a localized job database.

3. **Comprehensive Data Extraction**: Each job listing is parsed to extract 14 different data points including job details, company information, requirements, and responsibilities.


## Imported Libraries

The notebook relies on the following libraries:

- `requests`: to send HTTP requests to LinkedIn pages  
- `BeautifulSoup`: to parse and navigate HTML content  
- `pandas`: to structure scraped data and export it as CSV  
- `time`: to enforce delays and avoid rate limiting  
- `re`: to clean and normalize text fields  
- `urllib.parse`: to normalize URLs and extract job IDs  

In [1]:
import requests
from bs4 import BeautifulSoup
import csv
import pandas as pd
import time
import re
from typing import List, Dict
from urllib.parse import urljoin, urlparse, parse_qs, urlencode


## Architecture & Components

In [2]:
SEARCH_URL = "https://www.linkedin.com/jobs/search/?currentJobId=4336161261&f_TPR=r2592000&geoId=106155005&origin=JOB_SEARCH_PAGE_SEARCH_BUTTON&refresh=true&sortBy=DD"
SEARCH_URL = SEARCH_URL + "?locale=en_US"
MAX_JOBS = 100
DELAY_BETWEEN_REQUESTS = 2.0

In [3]:
base_url = requests.get(SEARCH_URL)
html = base_url.content
page = BeautifulSoup(html, 'lxml')
print(page.prettify())

<!DOCTYPE html>
<html lang="en">
 <head>
  <meta content="d_jobs_guest_search" name="pageKey"/>
  <meta content="max-image-preview:large, noarchive" name="robots"/>
  <meta content="max-image-preview:large, archive" name="bingbot"/>
  <!-- -->
  <meta content="urlType=jserp_custom;emptyResult=false" name="linkedin:pageTag"/>
  <meta content="en_US" name="locale"/>
  <!-- -->
  <meta data-app-version="2.0.2712" data-browser-id="270039e7-2b66-4996-8989-478eb15703a3" data-call-tree-id="AAZGf7OLqVoEKK31kmfQjw==" data-dfp-member-lix-treatment="control" data-disable-jsbeacon-pagekey-suffix="false" data-dna-member-lix-treatment="enabled" data-enable-page-view-heartbeat-tracking="" data-human-member-lix-treatment="enabled" data-is-epd-audit-event-enabled="false" data-is-feed-sponsored-tracking-kill-switch-enabled="false" data-member-id="0" data-multiproduct-name="jobs-guest-frontend" data-network-interceptor-lix-value="control" data-page-instance="urn:li:page:d_jobs_guest_search;Zmwjiuu3TEyO2B

In [4]:
# List of all search URLs to scrape
SEARCH_URLS = [
    "https://www.linkedin.com/jobs/search/?currentJobId=4336161261&f_TPR=r2592000&geoId=106155005&origin=JOB_SEARCH_PAGE_SEARCH_BUTTON&refresh=true&sortBy=DD",
    "https://www.linkedin.com/jobs/search/?currentJobId=4340701283&geoId=106155005&keywords=C%2B%2B%20Developer&origin=JOB_SEARCH_PAGE_SEARCH_BUTTON&refresh=true",
    "https://www.linkedin.com/jobs/search/?currentJobId=4340154665&geoId=106155005&keywords=IT%20Support%20Engineer&origin=JOB_SEARCH_PAGE_SEARCH_BUTTON&refresh=true",
    "https://www.linkedin.com/jobs/search/?currentJobId=4326267388&geoId=106155005&keywords=Network%20Administrator&origin=JOB_SEARCH_PAGE_SEARCH_BUTTON&refresh=true",
    "https://www.linkedin.com/jobs/search/?currentJobId=4315159411&geoId=106155005&keywords=NLP%20Engineer&origin=JOB_SEARCH_PAGE_SEARCH_BUTTON&refresh=true&start=25",
    "https://www.linkedin.com/jobs/search/?currentJobId=4325163431&geoId=106155005&keywords=DevOps%20Intern&origin=JOB_SEARCH_PAGE_SEARCH_BUTTON&refresh=true&start=25",
    "https://www.linkedin.com/jobs/search/?currentJobId=4335081605&geoId=106155005&keywords=Film%20Internship&origin=JOB_SEARCH_PAGE_SEARCH_BUTTON&refresh=true",
    "https://www.linkedin.com/jobs/search/?currentJobId=4344321369&geoId=106155005&keywords=Biomechanical%20Engineer&origin=JOB_SEARCH_PAGE_SEARCH_BUTTON&refresh=true",
    "https://www.linkedin.com/jobs/search/?currentJobId=4336707283&geoId=106155005&keywords=International%20Relations%20Intern&origin=JOB_SEARCH_PAGE_SEARCH_BUTTON&refresh=true",
    "https://www.linkedin.com/jobs/search/?currentJobId=4347747377&geoId=106155005&keywords=Energy%20Policy&origin=JOB_SEARCH_PAGE_SEARCH_BUTTON&refresh=true",
    "https://www.linkedin.com/jobs/search/?currentJobId=4343886669&geoId=106155005&keywords=Full%20Stack%20Developer&origin=JOB_SEARCH_PAGE_SEARCH_BUTTON&refresh=true",
    "https://www.linkedin.com/jobs/search/?currentJobId=4343886669&geoId=106155005&keywords=react%20Developer&origin=JOB_SEARCH_PAGE_SEARCH_BUTTON&refresh=true",
    "https://www.linkedin.com/jobs/search/?currentJobId=4348505294&geoId=106155005&keywords=Data%20Analyst&origin=JOB_SEARCH_PAGE_SEARCH_BUTTON&refresh=true",
    "https://www.linkedin.com/jobs/search/?currentJobId=4250907943&geoId=106155005&keywords=data%20scientist&origin=JOB_SEARCH_PAGE_KEYWORD_AUTOCOMPLETE&refresh=true",
    "https://www.linkedin.com/jobs/search/?currentJobId=4343876800&geoId=106155005&keywords=software%20engineer&origin=JOB_SEARCH_PAGE_KEYWORD_AUTOCOMPLETE&refresh=true&start=25",
    "https://www.linkedin.com/jobs/search/?currentJobId=4349022100&geoId=106155005&keywords=Mechatronics%20Engineer&origin=JOB_SEARCH_PAGE_SEARCH_BUTTON&refresh=true",
    "https://www.linkedin.com/jobs/search/?currentJobId=4349022100&geoId=106155005&keywords=Robotics%20Engineer&origin=JOB_SEARCH_PAGE_SEARCH_BUTTON&refresh=true",
    "https://www.linkedin.com/jobs/search/?currentJobId=4318859380&geoId=106155005&keywords=Mortgage%20Specialist&origin=JOB_SEARCH_PAGE_SEARCH_BUTTON&refresh=true",
    "https://www.linkedin.com/jobs/search/?currentJobId=4349743588&geoId=106155005&keywords=Project%20Manager&origin=JOB_SEARCH_PAGE_SEARCH_BUTTON&refresh=true",
    "https://www.linkedin.com/jobs/search/?currentJobId=4332770496&geoId=106155005&keywords=AI%20Engineer&origin=JOB_SEARCH_PAGE_SEARCH_BUTTON&refresh=true&sortBy=R&spellCorrectionEnabled=true",
    "https://www.linkedin.com/jobs/search/?currentJobId=4331304082&geoId=106155005&keywords=AI%2FML&origin=JOB_SEARCH_PAGE_SEARCH_BUTTON&refresh=true",
    "https://www.linkedin.com/jobs/search/?currentJobId=4350262635&f_TPR=r2592000&geoId=106155005&keywords=Digital%20Marketing%20Manager&origin=JOB_SEARCH_PAGE_SEARCH_BUTTON&refresh=true&sortBy=R&spellCorrectionEnabled=true",
    "https://www.linkedin.com/jobs/search/?currentJobId=4327923965&f_TPR=r2592000&geoId=106155005&keywords=Investment%20Banking%20Analyst&origin=JOB_SEARCH_PAGE_SEARCH_BUTTON&refresh=true&sortBy=DD",
    "https://www.linkedin.com/jobs/search/?currentJobId=4344344234&f_TPR=r2592000&geoId=106155005&keywords=Business%20Development%20Trainee&origin=JOB_SEARCH_PAGE_SEARCH_BUTTON&refresh=true&sortBy=DD",
    "https://www.linkedin.com/jobs/search/?currentJobId=4347732538&f_TPR=r2592000&geoId=106155005&keywords=finance&origin=JOB_SEARCH_PAGE_KEYWORD_AUTOCOMPLETE&refresh=true&sortBy=DD",
    "https://www.linkedin.com/jobs/search/?currentJobId=4344003420&f_TPR=r2592000&geoId=106155005&keywords=HR%20Coordinator&origin=JOB_SEARCH_PAGE_SEARCH_BUTTON&refresh=true&sortBy=DD",
    "https://www.linkedin.com/jobs/search/?currentJobId=4327924107&f_TPR=r2592000&geoId=106155005&keywords=Recruitment%20Specialist&origin=JOB_SEARCH_PAGE_SEARCH_BUTTON&refresh=true&sortBy=DD",
    "https://www.linkedin.com/jobs/search/?currentJobId=4303333844&f_TPR=r2592000&geoId=106155005&keywords=Marketing%20Executive&origin=JOB_SEARCH_PAGE_SEARCH_BUTTON&refresh=true&sortBy=DD",
    "https://www.linkedin.com/jobs/search/?currentJobId=4344514062&f_TPR=r2592000&geoId=106155005&keywords=Sales%20Associate&origin=JOB_SEARCH_PAGE_SEARCH_BUTTON&refresh=true&sortBy=DD"
]

In [5]:
MAX_JOBS_PER_SEARCH = 40
TOTAL_MAX_JOBS = 2000
DELAY_BETWEEN_REQUESTS = 2.0
DELAY_BETWEEN_SEARCHES = 3.0
OUTPUT_FILENAME = 'linkedin_jobs_egypt_combined.csv'

## HTTP Headers Configuration
Avoid detection as a scraping bot

Request English content specifically

Handle compressed responses

Maintain persistent connections for efficiency

In [6]:
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1"
}

In [7]:
# Storage for scraped data
all_jobs_data = []
seen_job_ids = set()  # To track duplicates

## Text Cleaning and Normalization

A utility function removes:
- Extra whitespace
- Line breaks
- Formatting noise

This step ensures that:
- Job descriptions are consistent
- NLP-free feature extraction is easier
- CV matching is more reliable

In [ ]:
def clean_text(text):
    """Clean and normalize text"""
    if not text:
        return ''
    # Remove extra whitespace and newlines
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

In [ ]:
def normalize_job_url(url):
    """Normalize job URL to extract base job ID"""
    try:
        # Extract job ID from URL
        parsed = urlparse(url)
        path_parts = parsed.path.split('/')

        if 'view' in path_parts:
            idx = path_parts.index('view')
            if idx + 1 < len(path_parts):
                job_id = path_parts[idx + 1]
                return job_id

        # Fallback: extract from query parameters
        query_params = parse_qs(parsed.query)
        if 'currentJobId' in query_params:
            return query_params['currentJobId'][0]

        return url  # Return original URL if can't extract ID
    except:
        return url

In [ ]:
def get_job_links_from_search(search_url, max_jobs):
    """Extract job listing links from a single search results page"""
    try:
        print(f"\nFetching search results from: {search_url}")

        # Add locale parameter if not present
        if "locale=" not in search_url:
            sep = "&" if "?" in search_url else "?"
            search_url = search_url + f"{sep}locale=en_US"

        response = requests.get(search_url, headers=HEADERS, timeout=15)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'lxml')

        job_links = []
        job_ids = set()  # Track job IDs for this search

        # Try multiple selectors for job links
        job_cards = soup.find_all('a', class_='base-card__full-link')

        if not job_cards:
            job_cards = soup.select('div.base-search-card a[href*="/jobs/view/"]')

        if not job_cards:
            job_cards = soup.select('a[href*="/jobs/view/"]')

        for card in job_cards:
            if len(job_links) >= max_jobs:
                break

            href = card.get('href')
            if href and '/jobs/view/' in href:
                base_link = href.split('?')[0]
                job_url = f"{base_link}?skipRedirect=true"
                job_url = job_url.replace("eg.linkedin.com", "www.linkedin.com")

                # Extract job ID for deduplication
                job_id = normalize_job_url(job_url)

                # Check if we've already seen this job
                if job_id not in seen_job_ids and job_id not in job_ids:
                    job_links.append(job_url)
                    job_ids.add(job_id)
                    seen_job_ids.add(job_id)

        print(f"Found {len(job_links)} new job links from this search")
        return job_links

    except Exception as e:
        print(f"Error fetching job links from {search_url}: {e}")
        return []

In [ ]:
def scrape_job_details(job_url):
    """
    Robust scraper for a single LinkedIn job posting.
    """
    try:
        # Ensure we request English version of the page
        if "locale=" not in job_url:
            sep = "&" if "?" in job_url else "?"
            job_url = job_url + f"{sep}locale=en_US"

        print(f"  Scraping: {job_url}")
        response = requests.get(job_url, headers=HEADERS, timeout=15)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, "lxml")

        job_data = {
            'url': job_url,
            'job_title': '',
            'company': '',
            'company_url': '',
            'location': '',
            'job_type': '',
            'job_function': '',
            'career_level': '',
            'salary': '',
            'posted_date': '',
            'industries': '',
            'summary': '',
            'responsibilities': '',
            'qualifications': ''
        }

        # BASIC FIELDS
        # Title
        title_elem = soup.find('h1', class_='top-card-layout__title') or soup.find('h1', class_='topcard__title')
        job_data['job_title'] = clean_text(title_elem.get_text()) if title_elem else job_data['job_title']

        # Company + company_url
        company_elem = soup.find('a', class_='topcard__org-name-link') or soup.find('a', class_='topcard__org-name-link topcard__flavor--link ember-view')
        if company_elem:
            job_data['company'] = clean_text(company_elem.get_text())
            job_data['company_url'] = company_elem.get('href', '')
        else:
            # fallback: sometimes company is in a span
            sp = soup.find('span', class_='topcard__flavor')
            if sp:
                job_data['company'] = clean_text(sp.get_text())

        # Location
        location_elem = soup.find('span', class_='topcard__flavor topcard__flavor--bullet') or soup.find('span', class_='topcard__flavor')
        job_data['location'] = clean_text(location_elem.get_text()) if location_elem else job_data['location']

        # Posted date
        posted_elem = soup.find('span', class_='posted-time-ago__text') or soup.find('span', class_='topcard__flavor--metadata')
        job_data['posted_date'] = clean_text(posted_elem.get_text()) if posted_elem else job_data['posted_date']

        # Salary - try a few common selectors
        salary_elem = soup.find('span', class_='salary') or soup.find(string=re.compile(r"AED|USD|\$|EGP|EUR"))
        if salary_elem:
            job_data['salary'] = clean_text(salary_elem.get_text() if hasattr(salary_elem, 'get_text') else str(salary_elem))

        # CRITERIA (seniority, employment type, job function, industries)
        header_map = {
            "seniority level": "career_level",
            "employment type": "job_type",
            "job function": "job_function",
            "industries": "industries",
            "مستوى الأقدمية": "career_level",
            "نوع التوظيف": "job_type",
            "المهام الوظيفية": "job_function",
            "الوظائف": "job_function",
            "المجالات": "industries",
            "مجالات": "industries",
        }

        criteria_list = soup.find('ul', class_='description__job-criteria-list')
        if criteria_list:
            items = criteria_list.find_all('li', class_='description__job-criteria-item')
            for item in items:
                header_tag = item.find('h3', class_='description__job-criteria-subheader')
                value_tag = item.find('span', class_='description__job-criteria-text')
                header_text = clean_text(header_tag.get_text()) if header_tag else ''
                value_text = clean_text(value_tag.get_text()) if value_tag else ''
                key = header_map.get(header_text.lower())
                if key:
                    if job_data.get(key):
                        existing = job_data[key].split(", ")
                        if value_text not in existing:
                            job_data[key] = job_data[key] + ", " + value_text
                    else:
                        job_data[key] = value_text
                else:
                    low = header_text.lower()
                    if "senior" in low or "entry" in low or "level" in low or "مستوى" in low:
                        job_data['career_level'] = value_text
                    elif "employment" in low or "type" in low or "وظيف" in low or "توظيف" in low:
                        job_data['job_type'] = value_text
                    elif "function" in low or "job" in low or "المهام" in low:
                        job_data['job_function'] = value_text
                    elif "industr" in low or "مجال" in low:
                        job_data['industries'] = value_text

        # DESCRIPTION / SUMMARY / RESPONSIBILITIES / QUALIFICATIONS
        description_div = soup.find('div', class_='show-more-less-html__markup') or soup.find('div', class_='description__text')
        all_text = ""
        if description_div:
            all_text = description_div.get_text("\n", strip=True)
        else:
            big = soup.find('section', class_='core-section-container description')
            if big:
                all_text = big.get_text("\n", strip=True)

        summary_lines = []
        responsibilities_lines = []
        qualifications_lines = []

        responsibility_keywords = [
            "responsibilities", "what you will do", "what you'll do", "duties",
            "key responsibilities", "role & responsibilities", "role and responsibilities",
            "what you'll be doing", "what you'll do", "what you will be doing"
        ]
        qualification_keywords = [
            "qualifications", "requirements", "what we are looking for",
            "skills", "what you bring", "who you are", "needed", "requirements and skills"
        ]

        lines = [clean_text(l) for l in all_text.split("\n") if clean_text(l)]
        current_section = "summary"
        for line in lines:
            low = line.lower()

            if any(kw in low for kw in responsibility_keywords):
                current_section = "responsibilities"
                continue
            if any(kw in low for kw in qualification_keywords):
                current_section = "qualifications"
                continue

            if current_section == "summary":
                if len(line) < 200 and len(summary_lines) < 6:
                    summary_lines.append(line)
                else:
                    summary_lines.append(line)
            elif current_section == "responsibilities":
                responsibilities_lines.append(line)
            elif current_section == "qualifications":
                qualifications_lines.append(line)

        def unique_join(lst):
            out = []
            for s in lst:
                if s not in out:
                    out.append(s)
            return "\n".join(out).strip()

        job_data['summary'] = unique_join(summary_lines)
        job_data['responsibilities'] = unique_join(responsibilities_lines)
        job_data['qualifications'] = unique_join(qualifications_lines)

        return job_data

    except Exception as e:
        print(f"  Error scraping {job_url}: {e}")
        return {
            'url': job_url,
            'job_title': 'ERROR',
            'company': '',
            'company_url': '',
            'location': '',
            'job_type': '',
            'job_function': '',
            'career_level': '',
            'salary': '',
            'posted_date': '',
            'industries': '',
            'summary': str(e),
            'responsibilities': '',
            'qualifications': ''
        }

In [ ]:
# Main scraping loop
print(f"Starting to scrape from {len(SEARCH_URLS)} search URLs...")
print(f"Maximum jobs per search: {MAX_JOBS_PER_SEARCH}")
print(f"Overall maximum jobs: {TOTAL_MAX_JOBS}")
print("-" * 50)

for search_idx, search_url in enumerate(SEARCH_URLS, 1):
    if len(all_jobs_data) >= TOTAL_MAX_JOBS:
        print(f"\nReached overall maximum of {TOTAL_MAX_JOBS} jobs. Stopping.")
        break

    print(f"\n[{search_idx}/{len(SEARCH_URLS)}] Processing search URL...")

    # Get job links from this search
    job_links = get_job_links_from_search(search_url, MAX_JOBS_PER_SEARCH)

    if not job_links:
        print("  No job links found, skipping...")
        time.sleep(DELAY_BETWEEN_SEARCHES)
        continue

    # Scrape each job
    for i, job_url in enumerate(job_links, 1):
        if len(all_jobs_data) >= TOTAL_MAX_JOBS:
            break

        print(f"  [{i}/{len(job_links)}] Processing job...")

        job_data = scrape_job_details(job_url)
        all_jobs_data.append(job_data)

        if i < len(job_links):
            time.sleep(DELAY_BETWEEN_REQUESTS)

    # Wait between searches to avoid rate limiting
    if search_idx < len(SEARCH_URLS):
        print(f"  Waiting {DELAY_BETWEEN_SEARCHES} seconds before next search...")
        time.sleep(DELAY_BETWEEN_SEARCHES)

print(f"\n{'='*60}")
print(f"Scraping completed!")
print(f"Total unique jobs scraped: {len(all_jobs_data)}")
print(f"Total duplicate jobs skipped: {len(seen_job_ids) - len(all_jobs_data)}")
print(f"{'='*60}")

Starting to scrape from 29 search URLs...
Maximum jobs per search: 40
Overall maximum jobs: 2000
--------------------------------------------------

[1/29] Processing search URL...

Fetching search results from: https://www.linkedin.com/jobs/search/?currentJobId=4336161261&f_TPR=r2592000&geoId=106155005&origin=JOB_SEARCH_PAGE_SEARCH_BUTTON&refresh=true&sortBy=DD
✓ Found 7 new job links from this search
  [1/7] Processing job...
  Scraping: https://www.linkedin.com/jobs/view/office-administrator-at-fawry-msme-finance-4327729348?skipRedirect=true&locale=en_US
  [2/7] Processing job...
  Scraping: https://www.linkedin.com/jobs/view/data-entry-clerk-at-aujan-coca-cola-beverages-company-accbc-4341270778?skipRedirect=true&locale=en_US
  [3/7] Processing job...
  Scraping: https://www.linkedin.com/jobs/view/cabin-crew-cairo-at-air-arabia-4341068993?skipRedirect=true&locale=en_US
  [4/7] Processing job...
  Scraping: https://www.linkedin.com/jobs/view/accountant-at-fawry-msme-finance-432792012

In [ ]:
# Create DataFrame
columns = [
    "url", "job_title", "company", "company_url", "location",
    "job_type", "job_function", "career_level", "salary",
    "posted_date", "industries", "summary", "responsibilities", "qualifications"
]

In [ ]:
if all_jobs_data:
    df = pd.DataFrame(all_jobs_data)

    # Ensure all columns exist
    for col in columns:
        if col not in df.columns:
            df[col] = ''

    df = df[columns]

    # Final deduplication by URL
    df = df.drop_duplicates(subset=['url'])

    # Save to CSV
    df.to_csv(OUTPUT_FILENAME, index=False, encoding='utf-8-sig')
    print(f"\nData saved to: {OUTPUT_FILENAME}")

    # Print summary statistics
    print(f"\n{'='*60}")
    print("SUMMARY STATISTICS:")
    print(f"{'='*60}")
    print(f"Total jobs scraped: {len(df)}")
    print(f"Jobs with title: {df['job_title'].notna().sum()}")
    print(f"Jobs with company: {df['company'].notna().sum()}")
    print(f"Jobs with location: {df['location'].notna().sum()}")
    print(f"Jobs with job_type: {(df['job_type'] != '').sum()}")
    print(f"Jobs with career_level: {(df['career_level'] != '').sum()}")
    print(f"Jobs with job_function: {(df['job_function'] != '').sum()}")
    print(f"Jobs with industries: {(df['industries'] != '').sum()}")
    print(f"Jobs with responsibilities: {(df['responsibilities'] != '').sum()}")
    print(f"Jobs with qualifications: {(df['qualifications'] != '').sum()}")

    # Display first few jobs
    if len(df) > 0:
        print(f"\n{'='*60}")
        print("SAMPLE JOB (first entry):")
        print(f"{'='*60}")
        sample = df.iloc[0]
        for col in columns:
            value = sample[col]
            if col in ['summary', 'responsibilities', 'qualifications']:
                display_value = str(value)[:200] + "..." if len(str(value)) > 200 else value
            else:
                display_value = value
            print(f"\n{col.upper()}:")
            print(f"  {display_value}")
else:
    print("\nNo jobs were scraped. Check your search URLs and network connection.")


Data saved to: linkedin_jobs_egypt_combined.csv

SUMMARY STATISTICS:
Total jobs scraped: 287
Jobs with title: 287
Jobs with company: 287
Jobs with location: 287
Jobs with job_type: 287
Jobs with career_level: 287
Jobs with job_function: 287
Jobs with industries: 287
Jobs with responsibilities: 165
Jobs with qualifications: 248

SAMPLE JOB (first entry):

URL:
  https://www.linkedin.com/jobs/view/office-administrator-at-fawry-msme-finance-4327729348?skipRedirect=true&locale=en_US

JOB_TITLE:
  Office Administrator

COMPANY:
  Fawry MSME Finance

COMPANY_URL:
  https://eg.linkedin.com/company/fawry-msme-finance?trk=public_jobs_topcard-org-name

LOCATION:
  Cairo, Cairo, Egypt

JOB_TYPE:
  Full-time

JOB_FUNCTION:
  Administrative

CAREER_LEVEL:
  Executive

SALARY:
  

POSTED_DATE:
  1 day ago

INDUSTRIES:
  Financial Services

SUMMARY:
  Oversee daily administrative operations related to the building and office facilities
Coordinate with service providers (maintenance, cleaning, securi

## Role in the CV–Job Matching System

This dataset is designed to be matched against a **CV database** using features such as:

- Skill overlap
- Career level compatibility
- Domain similarity
- Responsibility alignment

The scraped jobs serve as **ground-truth job representations**, enabling:
- Job ranking
- Candidate recommendation
- Supervised or semi-supervised learning
